In [ ]:
import re
import jieba
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Define base path for the project
BASE = Path('.')  # Change this to your actual project path if needed

In [ ]:
# Load your lyrics data
df = pd.read_csv("lyrics_chunks_enriched.csv")
print(f"Loaded data with shape: {df.shape}")
print(df.head())

In [ ]:
# Setup stopwords
STOP = BASE/"stopwords-zh.txt"
stopwords = set()
if STOP.exists():
    stopwords = {w.strip() for w in open(STOP, encoding="utf-8") if w.strip()}
else:
    print("Warning: Stopwords file not found. Creating an empty stopword list.")

# Additional manual stopwords
extra_stop = {"的","了","是","在","就","都","和","也","与","而","及","还有",
              "一个","没有","我们","你们","他们","什么","怎么","因为","然后",
              "啦","啊","呀","嘛","吧","呢","哦","喔","哈","嗯","呜","啦啦"}
stopwords |= extra_stop

# Single character whitelist (words to keep)
single_char_whitelist = {"爱","梦","心","夜","雨","花","海","光","路","风","歌","酒"}

print(f"Total stopwords: {len(stopwords)}")
print(f"Single character whitelist: {single_char_whitelist}")

In [ ]:
def zh_tokenize(s: str):
    """
    Tokenize Chinese text with various filtering rules:
    - Remove whitespace
    - Segment with Jieba
    - Filter stopwords, numbers, Latin characters, punctuation
    - Filter single characters (except whitelist)
    """
    s = re.sub(r"\s+", "", str(s))
    toks = [w for w in jieba.lcut(s, HMM=True) if w]
    out = []
    for w in toks:
        # Filter: stopwords, pure numbers/Latin, punctuation
        if w in stopwords:
            continue
        if re.fullmatch(r"[0-9A-Za-z]+", w):
            continue
        if re.fullmatch(r"\W+", w):
            continue
        # Filter most single characters (keep those in whitelist)
        if len(w)==1 and w not in single_char_whitelist:
            continue
        out.append(w)
    return out

In [ ]:
# Group lyrics by artist if needed
# If your data doesn't have a cluster column already, create one using artist
if 'artist' in df.columns:
    # Group by artist
    artist_lyrics = df.groupby('artist')['text'].apply(lambda x: ' '.join(x)).reset_index()
    artist_lyrics.columns = ['artist', 'text']

    # Create a cluster column based on artist
    artist_lyrics['cluster'] = np.arange(len(artist_lyrics))

    docs_by_cluster = artist_lyrics
    print(f"Created {len(docs_by_cluster)} artist clusters")
else:
    # Assume the data already has clusters
    docs_by_cluster = df
    print(f"Using existing clusters, total: {df['cluster'].nunique()}")

print(docs_by_cluster.head())

In [ ]:
# Process each document and create a dataframe with all segmented words
all_words_list = []

for idx, row in docs_by_cluster.iterrows():
    text = row['text']
    cluster = row['cluster']
    artist = row['artist'] if 'artist' in row else f"Cluster_{cluster}"

    # Segment the text
    words = zh_tokenize(text)

    # Add each word as a separate row
    for word in words:
        all_words_list.append({
            'cluster': cluster,
            'artist': artist,
            'word': word
        })

# Create dataframe with all segmented words (one word per line)
all_words_df = pd.DataFrame(all_words_list)
print(f"Created dataframe with {len(all_words_df)} segmented words")
print(all_words_df.head())

# Save to CSV (one word per line)
all_words_df.to_csv("segmented_words.csv", index=False)
print("Saved segmented words to 'segmented_words.csv'")

In [ ]:
# Count word frequencies overall
word_counts = all_words_df['word'].value_counts().reset_index()
word_counts.columns = ['word', 'frequency']

print(f"Found {len(word_counts)} unique words")
print("\nTop 20 most frequent words:")
print(word_counts.head(20))

# Save frequency dataset
word_counts.to_csv("word_frequencies.csv", index=False)
print("Saved word frequencies to 'word_frequencies.csv'")

In [ ]:
# Get documents and clusters
clusters = docs_by_cluster["cluster"].astype(int).tolist()
docs = docs_by_cluster["text"].tolist()
C = len(docs)  # Number of clusters

# Vectorize with CountVectorizer
vect = CountVectorizer(tokenizer=zh_tokenize, ngram_range=(1,2), min_df=2)
X = vect.fit_transform(docs)      # Shape: [C, V]
terms = np.array(vect.get_feature_names_out())
T = X.toarray().astype("float32") # Count matrix

# Calculate c-TF-IDF
tf = (T / (T.sum(axis=1, keepdims=True) + 1e-9))           # [C, V]
df_c = (T > 0).sum(axis=0)                                  # How many clusters each token appears in
idf = np.log((C + 1) / (df_c + 1)) + 1.0                   # Smoothed IDF
ctfidf = tf * idf                                           # [C, V]

# Get top terms for each cluster
top_n = 10  # Number of top terms to extract per cluster
cluster_terms = []

for i in range(C):
    # Sort terms by c-TF-IDF scores
    top_indices = ctfidf[i].argsort()[-top_n:][::-1]
    top_terms = [(terms[idx], ctfidf[i][idx]) for idx in top_indices]

    cluster_label = docs_by_cluster.iloc[i]['artist'] if 'artist' in docs_by_cluster.columns else f"Cluster {i}"

    cluster_terms.append({
        'cluster': i,
        'label': cluster_label,
        'terms': top_terms
    })

    print(f"\nTop terms for {cluster_label}:")
    for term, score in top_terms:
        print(f"  {term}: {score:.4f}")

In [ ]:
# Plot top 20 most frequent words
plt.figure(figsize=(12, 6))
top_words = word_counts.head(20)
plt.barh(top_words['word'][::-1], top_words['frequency'][::-1])
plt.xlabel('Frequency')
plt.ylabel('Words')
plt.title('Top 20 Most Frequent Words')
plt.tight_layout()
plt.show()

# Save cluster terms to CSV
cluster_terms_rows = []
for cluster in cluster_terms:
    cluster_id = cluster['cluster']
    label = cluster['label']
    for term, score in cluster['terms']:
        cluster_terms_rows.append({
            'cluster': cluster_id,
            'label': label,
            'term': term,
            'ctfidf': score
        })

cluster_terms_df = pd.DataFrame(cluster_terms_rows)
cluster_terms_df.to_csv("cluster_top_terms.csv", index=False)
print("Saved cluster top terms to 'cluster_top_terms.csv'")